In [1]:
import os
import cv2
import threading
import numpy as np
import time
from collections import defaultdict
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse, HTMLResponse, JSONResponse
from insightface.app import FaceAnalysis

# ================== CONFIG ==================
RTSP_URL  = "rtsp://admin:@192.168.1.69:554/user=admin&password=&channel=1&stream=0.sdp"
EMB_DIR   = "data/embeddings"
ROI_PATH  = "roi.npy"          # file lưu tọa độ ROI
DATASET_DIR   = "dataset_test"
SIM_THRESHOLD = 0.3
DET_THRESHOLD = 0.6
MIN_FACE_PX   = 60             # face width tối thiểu (px) trong ROI đã resize
# ============================================

## Load Face DB
Đọc toàn bộ file `.npy` trong `data/embeddings/`.

Mỗi file = 1 người, tên file = tên người, nội dung = embedding vector 512 chiều đã normalize.

`np.stack` gom list of vector thành matrix `(N, 512)` để tính cosine similarity bằng 1 phép nhân ma trận duy nhất thay vì loop.

In [2]:
print("chao xin")

chao xin


In [3]:
def reload_db():
    """Load/Reload toàn bộ DB embedding từ disk vào RAM.
    Gọi lại sau mỗi lần đăng ký ảnh mới để cập nhật nhận diện ngay.
    """
    global db_names, db_embs
    person_embs = defaultdict(list)

    for fname in os.listdir(EMB_DIR):
        if not fname.endswith(".npy"):
            continue
        name = fname.rsplit("_", 1)[0]          # anhht_1.npy → anhht
        emb  = np.load(os.path.join(EMB_DIR, fname))
        emb  = emb / np.linalg.norm(emb)        # đảm bảo L2-norm = 1
        person_embs[name].append(emb)

    _names, _embs = [], []
    for name, embs in person_embs.items():
        proto = np.stack(embs).mean(axis=0)     # mean embedding
        proto = proto / np.linalg.norm(proto)   # re-normalize
        _names.append(name)
        _embs.append(proto)

    db_names = _names
    db_embs  = np.stack(_embs) if _embs else np.zeros((0, 512))
    print(f"[DB] Loaded {len(db_names)} người: {db_names}")
    print(f"[DB] db_embs shape: {db_embs.shape}")

db_names = []
db_embs  = np.zeros((0, 512))
reload_db()

[DB] Loaded 1 người: ['anhht']
[DB] db_embs shape: (1, 512)


In [4]:
# ── Cell: Load DB từ SQLite + FAISS (thay thế reload_db cũ) ──
# Chạy cell này thay cho reload_db() để dùng embedding từ app đăng ký mới
# Đường dẫn tới shared data của face_registration app
import sqlite3, json
import numpy as np

SQLITE_PATH  = "/home/anhht/face_registration/data/employees.db"
FAISS_PATH   = "/home/anhhtface-registration/data/faiss.index"
ID_MAP_PATH  = "/home/anhht/face-registration/data/id_map.json"
EMBEDDING_DIM = 512

def reload_db_from_sqlite():
    """
    Load toàn bộ embeddings từ SQLite thay vì .npy files.
    Ghi đè db_names và db_embs — recognize() không cần sửa.
    
    Chiến lược: mean pooling per person giống cách cũ
    → 1 proto vector per person → matrix (N_person, 512)
    """
    global db_names, db_embs

    conn = sqlite3.connect(SQLITE_PATH)
    rows = conn.execute(
        "SELECT employee_id, embedding FROM face_embeddings ORDER BY employee_id"
    ).fetchall()
    
    # Lấy tên hiển thị từ bảng employees
    emp_rows = conn.execute("SELECT employee_id, name FROM employees").fetchall()
    conn.close()
    
    id_to_name = {r[0]: r[1] for r in emp_rows}

    if not rows:
        db_names = []
        db_embs  = np.zeros((0, EMBEDDING_DIM), dtype=np.float32)
        print("[DB] Trống — chưa có nhân viên nào đăng ký")
        return

    # Gom embeddings theo employee_id
    from collections import defaultdict
    person_embs = defaultdict(list)
    for employee_id, blob in rows:
        emb = np.frombuffer(blob, dtype=np.float32).copy()
        person_embs[employee_id].append(emb)

    # Mean pooling + re-normalize → 1 proto vector per person
    names, protos = [], []
    for employee_id, embs in person_embs.items():
        stack = np.stack(embs)           # (K, 512)
        mean  = stack.mean(axis=0)       # (512,)
        proto = mean / (np.linalg.norm(mean) + 1e-8)
        display_name = id_to_name.get(employee_id, employee_id)
        names.append(display_name)
        protos.append(proto)

    db_names = names
    db_embs  = np.stack(protos).astype(np.float32)  # (N_person, 512)
    print(f"[DB] Loaded {len(db_names)} người từ SQLite:")
    for n in db_names:
        print(f"  - {n}")

# Chạy ngay
reload_db_from_sqlite()

# Ghi đè hàm reload_db cũ để /reload endpoint vẫn hoạt động
reload_db = reload_db_from_sqlite

[DB] Loaded 2 người từ SQLite:
  - Hoàng Tuấn Anh
  - tainm


In [5]:
import sqlite3
import numpy as np

SQLITE_PATH  = "/home/anhht/face_registration/data/employees.db"
EMBEDDING_DIM = 512

def reload_db_from_sqlite():
    """
    Load embeddings từ SQLite nhưng KHÔNG dùng mean pooling.
    Mỗi embedding = 1 vector trong database.
    """

    global db_names, db_embs

    conn = sqlite3.connect(SQLITE_PATH)

    rows = conn.execute(
        "SELECT employee_id, embedding FROM face_embeddings ORDER BY employee_id"
    ).fetchall()

    emp_rows = conn.execute(
        "SELECT employee_id, name FROM employees"
    ).fetchall()

    conn.close()

    id_to_name = {r[0]: r[1] for r in emp_rows}

    if not rows:
        db_names = []
        db_embs = np.zeros((0, EMBEDDING_DIM), dtype=np.float32)
        print("[DB] Trống — chưa có nhân viên")
        return

    names = []
    embs  = []

    for employee_id, blob in rows:

        emb = np.frombuffer(blob, dtype=np.float32).copy()

        # normalize embedding
        emb = emb / (np.linalg.norm(emb) + 1e-8)

        display_name = id_to_name.get(employee_id, employee_id)

        names.append(display_name)
        embs.append(emb)

    db_names = names
    db_embs  = np.stack(embs).astype(np.float32)

    print(f"[DB] Loaded {len(db_names)} embeddings từ SQLite")
    print(f"[DB] Tương ứng {len(set(db_names))} người")

    for n in set(db_names):
        print(f"  - {n}")


# chạy
reload_db_from_sqlite()

reload_db = reload_db_from_sqlite

[DB] Loaded 22 embeddings từ SQLite
[DB] Tương ứng 2 người
  - Hoàng Tuấn Anh
  - tainm


## Cell 3 — Load Face Model
`buffalo_l` = backbone R100, detector RetinaFace.

`ctx_id=0` = GPU 0. Đổi thành `ctx_id=-1` nếu muốn chạy CPU.

`det_size=(640,640)` = kích thước ảnh detector nhận vào bên trong model.
Frame đầu vào sẽ được scale về 640x640 bên trong InsightFace trước khi detect.

In [6]:
face_app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
face_app.prepare(ctx_id=0, det_size=(640, 640))
print("Face model loaded")

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'use_tf32': '1', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /home/anhht/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 

## Cell 4 — Camera Reader (Threading)

**Vấn đề nếu không dùng thread riêng:**
```
gen_frames(): cap.read() → detect(80ms) → encode → gửi → cap.read() ...
```
Trong 80ms detect, camera vẫn gửi frame vào buffer.
Buffer tích lũy → khi đọc lại thì frame đã cũ 500ms → lag.

**Giải pháp: tách thread đọc camera độc lập**
```
Thread 1: cap.read() liên tục → giữ frame mới nhất vào latest_frame
Thread 2: lấy latest_frame → detect → encode → gửi
```
Thread 2 dù mất 80ms, Thread 1 vẫn đọc frame mới → luôn xử lý frame mới nhất.

**Lock** bảo vệ `latest_frame` khỏi race condition:
Thread 1 đang ghi, Thread 2 không được đọc cùng lúc và ngược lại.

In [7]:
latest_frame = None
frame_lock   = threading.Lock()

def camera_reader():
    """Thread liên tục đọc frame mới nhất từ RTSP, tránh lag buffer."""
    cap = cv2.VideoCapture(RTSP_URL, cv2.CAP_FFMPEG)
    global latest_frame
    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            cap.open(RTSP_URL)
            continue
        if frame.ndim != 3 or frame.shape[0] < 10 or frame.shape[1] < 10:
            continue
        with frame_lock:
            latest_frame = frame.copy()
    cap.release()
    

cam_thread = threading.Thread(target=camera_reader, daemon=True)
cam_thread.start()
print("Camera thread started")

Camera thread started


## Cell 5 — ROI Helpers

**Load ROI:** đọc tọa độ đã lưu từ `/roi/select`.

**recognize:** tính cosine similarity giữa embedding đầu vào và toàn bộ DB.
`db_embs @ face_emb` = matrix-vector multiplication → ra vector similarity (N,).
Lấy argmax = người có similarity cao nhất.

**scale_bbox_to_full:** bbox detect trên ROI resized (640x480) cần scale ngược
về tọa độ frame gốc để vẽ đúng vị trí.
```
x_goc = roi_x1 + x_resized × (roi_width / resize_width)
```

In [8]:
# Load ROI nếu đã có từ lần trước
roi_coords = None
if os.path.exists(ROI_PATH):
    roi_coords = np.load(ROI_PATH).tolist()   # [x1, y1, x2, y2]
    print(f"ROI loaded: {roi_coords}")
else:
    print("Chưa có ROI — vào /roi/select để chọn")


def recognize(face_emb: np.ndarray):
    """
    So sánh embedding với toàn bộ DB (đã mean-pooled).
    Trả về (tên, score) nếu vượt threshold, ("Unknown", score) nếu không.
    """
    if len(db_names) == 0:
        return "Unknown", 0.0
    sims  = db_embs @ face_emb               # (N_person,) cosine similarity
    idx   = int(np.argmax(sims))
    score = float(sims[idx])
    name  = db_names[idx] if score >= SIM_THRESHOLD else "Unknown"
    return name, score


def scale_bbox_to_frame(x1, y1, x2, y2, roi_w, roi_h):
    """Map tọa độ bbox từ không gian ROI-resized về không gian frame gốc."""
    rx1, ry1, rx2, ry2 = roi_coords
    sx = (rx2 - rx1) / roi_w
    sy = (ry2 - ry1) / roi_h
    return (
        int(rx1 + x1 * sx),
        int(ry1 + y1 * sy),
        int(rx1 + x2 * sx),
        int(ry1 + y2 * sy),
    )

ROI loaded: [922, 232, 1342, 812]


## Cell 6 — FastAPI App + Tất cả Endpoints

**4 endpoints:**
- `GET /roi/select` — UI browser kéo chuột chọn ROI
- `GET /snapshot`   — 1 frame JPEG từ camera (dùng cho UI)
- `POST /roi/save`  — nhận tọa độ, lưu roi.npy, cập nhật roi_coords live
- `GET /roi/current`— kiểm tra ROI đang dùng
- `GET /video`      — MJPEG stream với bbox nhận diện

**gen_frames pipeline:**
```
get_latest_frame()       ← lấy từ thread đọc camera
    ↓
crop ROI + resize        ← tăng face size
    ↓
face_app.get()           ← detect + landmark + embedding
    ↓
filter: det_score, face_w ← bỏ detection kém
    ↓
recognize()              ← cosine similarity
    ↓
scale_bbox_to_full()     ← vẽ đúng vị trí trên frame gốc
    ↓
imencode + yield         ← gửi MJPEG chunk
```

In [9]:
app = FastAPI()

# ─────────────────────────────────────────────
#  SNAPSHOT + ROI
# ─────────────────────────────────────────────

@app.get("/snapshot")
def snapshot():
    """Trả về 1 frame JPEG để UI dùng làm nền chọn ROI."""
    with frame_lock:
        frame = latest_frame.copy() if latest_frame is not None else None
    if frame is None:
        return JSONResponse({"error": "no frame"}, status_code=503)
    _, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return StreamingResponse(iter([buf.tobytes()]), media_type="image/jpeg")


@app.get("/roi/select", response_class=HTMLResponse)
def roi_select_ui():
    """Giao diện web kéo-thả để chọn ROI trực tiếp trên frame."""
    return """<!DOCTYPE html>
<html>
<head>
  <title>Chọn ROI</title>
  <style>
    body { margin:0; background:#111; display:flex; flex-direction:column;
           align-items:center; padding:20px; font-family:monospace; color:#eee; }
    canvas { cursor:crosshair; border:2px solid #0f0; }
    #info  { margin:10px; font-size:14px; }
    button { margin:8px; padding:8px 20px; background:#0a0; color:#fff;
             border:none; cursor:pointer; font-size:14px; }
  </style>
</head>
<body>
  <h3>Kéo để chọn vùng ROI (cửa ra vào)</h3>
  <canvas id="c"></canvas>
  <div id="info">Đang tải frame...</div>
  <button onclick="saveROI()">Lưu ROI</button>
  <button onclick="location.reload()">Tải lại frame</button>
<script>
const canvas = document.getElementById('c');
const ctx    = canvas.getContext('2d');
let img = new Image();
let sx, sy, ex, ey, drawing = false;
let scaleX = 1, scaleY = 1;

img.onload = () => {
  const maxW = Math.min(window.innerWidth - 40, 1200);
  scaleX = img.naturalWidth  / Math.min(img.naturalWidth, maxW);
  scaleY = img.naturalHeight / (img.naturalHeight * maxW / img.naturalWidth);
  canvas.width  = Math.min(img.naturalWidth,  maxW);
  canvas.height = img.naturalHeight * canvas.width / img.naturalWidth;
  ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
  document.getElementById('info').textContent =
    `Frame: ${img.naturalWidth}×${img.naturalHeight}`;
};
img.src = '/snapshot?' + Date.now();

canvas.addEventListener('mousedown', e => {
  const r = canvas.getBoundingClientRect();
  sx = e.clientX - r.left; sy = e.clientY - r.top; drawing = true;
});
canvas.addEventListener('mousemove', e => {
  if (!drawing) return;
  const r = canvas.getBoundingClientRect();
  ex = e.clientX - r.left; ey = e.clientY - r.top;
  ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
  ctx.strokeStyle = '#0f0'; ctx.lineWidth = 2;
  ctx.strokeRect(sx, sy, ex-sx, ey-sy);
});
canvas.addEventListener('mouseup', () => { drawing = false; });

function saveROI() {
  if (sx === undefined) { alert('Chưa chọn vùng!'); return; }
  const x1 = Math.round(Math.min(sx,ex)*scaleX), y1 = Math.round(Math.min(sy,ey)*scaleY);
  const x2 = Math.round(Math.max(sx,ex)*scaleX), y2 = Math.round(Math.max(sy,ey)*scaleY);
  fetch('/roi/save', {
    method:'POST', headers:{'Content-Type':'application/json'},
    body: JSON.stringify({x1,y1,x2,y2})
  }).then(r=>r.json()).then(d => {
    document.getElementById('info').textContent =
      `Đã lưu ROI: [${d.x1}, ${d.y1}, ${d.x2}, ${d.y2}]`;
  });
}
</script>
</body>
</html>"""


@app.post("/roi/save")
async def roi_save(request: Request):
    global roi_coords
    data = await request.json()
    roi_coords = [data["x1"], data["y1"], data["x2"], data["y2"]]
    np.save(ROI_PATH, np.array(roi_coords))
    return roi_coords


@app.get("/roi/current")
def roi_current():
    return {"roi": roi_coords}


# ─────────────────────────────────────────────
#  VIDEO STREAM
# ─────────────────────────────────────────────

def gen_frames():
    """Generator MJPEG — crop ROI, resize, detect, recognize, scale bbox về frame gốc."""
    while True:
        with frame_lock:
            frame = latest_frame.copy() if latest_frame is not None else None
        if frame is None:
            continue

        draw_frame = frame.copy()

        if roi_coords is not None:
            rx1, ry1, rx2, ry2 = roi_coords
            h, w = frame.shape[:2]
            rx1, ry1 = max(0, rx1), max(0, ry1)
            rx2, ry2 = min(w, rx2), min(h, ry2)

            roi          = frame[ry1:ry2, rx1:rx2]
            roi_resized  = cv2.resize(roi, (640, 480), interpolation=cv2.INTER_LINEAR)
            faces        = face_app.get(roi_resized)

            # Vẽ khung ROI
            cv2.rectangle(draw_frame, (rx1,ry1), (rx2,ry2), (255,165,0), 2)
            cv2.putText(draw_frame, "ROI", (rx1, ry1-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,165,0), 1)

            for face in faces:
                if face.det_score < DET_THRESHOLD:
                    continue
                fx1, fy1, fx2, fy2 = map(int, face.bbox)
                if (fx2 - fx1) < MIN_FACE_PX:
                    continue
                name, score = recognize(face.normed_embedding)
                bx1,by1,bx2,by2 = scale_bbox_to_frame(fx1,fy1,fx2,fy2, 640, 480)
                color = (0,255,0) if name != "Unknown" else (0,0,255)
                cv2.rectangle(draw_frame, (bx1,by1), (bx2,by2), color, 2)
                cv2.putText(draw_frame, f"{name} {score:.2f}",
                            (bx1, by1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        else:
            faces = face_app.get(frame)
            for face in faces:
                if face.det_score < DET_THRESHOLD:
                    continue
                x1,y1,x2,y2 = map(int, face.bbox)
                name, score = recognize(face.normed_embedding)
                color = (0,255,0) if name != "Unknown" else (0,0,255)
                cv2.rectangle(draw_frame, (x1,y1), (x2,y2), color, 2)
                cv2.putText(draw_frame, f"{name} {score:.2f}",
                            (x1,y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            cv2.putText(draw_frame, "ROI chua duoc chon - vao /roi/select",
                        (20,40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,165,255), 2)

        # Overlay REC nếu đang ghi
        if recording_state["active"]:
            cv2.putText(draw_frame, "● REC",
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 2)
            cv2.putText(draw_frame, str(recording_state["filename"]),
                        (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)

        _, buf = cv2.imencode(".jpg", draw_frame, [cv2.IMWRITE_JPEG_QUALITY, 75])
        yield (b"--frame\r\nContent-Type: image/jpeg\r\n\r\n"
               + buf.tobytes() + b"\r\n")


@app.get("/video")
def video():
    return StreamingResponse(gen_frames(),
                             media_type="multipart/x-mixed-replace; boundary=frame")


# ─────────────────────────────────────────────
#  ĐĂNG KÝ KHUÔN MẶT
# ─────────────────────────────────────────────

@app.get("/register", response_class=HTMLResponse)
def register_ui():
    """Web UI đăng ký khuôn mặt nhân viên."""
    return """<!DOCTYPE html>
<html>
<head>
  <title>Đăng ký khuôn mặt</title>
  <style>
    * { box-sizing:border-box; margin:0; padding:0; }
    body { background:#0d0d0d; color:#eee; font-family:monospace;
           display:flex; flex-direction:column; align-items:center; padding:24px; }
    h2   { margin-bottom:16px; color:#0f0; }
    .card { background:#1a1a1a; border:1px solid #333; border-radius:8px;
             padding:20px; width:100%; max-width:720px; margin-bottom:16px; }
    input { width:100%; padding:8px; background:#222; border:1px solid #555;
            color:#eee; border-radius:4px; font-size:15px; }
    .row  { display:flex; gap:10px; margin-top:10px; }
    button { flex:1; padding:10px; border:none; border-radius:4px;
             cursor:pointer; font-size:14px; font-family:monospace; }
    .btn-capture { background:#0a0; color:#fff; }
    .btn-reload  { background:#555; color:#fff; }
    .btn-delete  { background:#a00; color:#fff; }
    #preview { width:100%; border-radius:4px; margin-top:12px;
               border:1px solid #333; display:none; }
    #log { background:#111; border:1px solid #333; border-radius:4px;
           padding:12px; font-size:13px; max-height:220px;
           overflow-y:auto; white-space:pre-wrap; margin-top:12px; }
    .ok  { color:#0f0; } .err { color:#f55; }
    table { width:100%; border-collapse:collapse; font-size:13px; }
    th,td { padding:6px 10px; border-bottom:1px solid #2a2a2a; text-align:left; }
    th { color:#0f0; }
  </style>
</head>
<body>
  <h2>Đăng ký khuôn mặt nhân viên</h2>
  <div class="card">
    <b>Bước 1 — Đứng trước camera, điền tên rồi bấm Chụp (3–5 lần, đổi góc mỗi lần)</b>
    <div class="row" style="margin-top:10px">
      <input id="name" placeholder="Nhập tên nhân viên (vd: anhht)" />
    </div>
    <div class="row">
      <button class="btn-capture" onclick="capture()">📸 Chụp ảnh</button>
      <button class="btn-reload"  onclick="loadList()">🔄 Tải danh sách</button>
    </div>
    <img id="preview" />
    <div id="log">Sẵn sàng...</div>
  </div>
  <div class="card">
    <b>Danh sách đã đăng ký</b>
    <div id="list-area" style="margin-top:10px">Bấm 🔄 để tải...</div>
  </div>
<script>
function log(msg, ok=true) {
  const el = document.getElementById('log');
  const line = document.createElement('div');
  line.className = ok ? 'ok' : 'err';
  line.textContent = new Date().toLocaleTimeString() + '  ' + msg;
  el.appendChild(line);
  el.scrollTop = el.scrollHeight;
}
async function capture() {
  const name = document.getElementById('name').value.trim();
  if (!name) { log('⚠ Chưa nhập tên!', false); return; }
  log(`Đang chụp cho "${name}"...`);
  try {
    const r = await fetch('/capture/' + encodeURIComponent(name), {method:'POST'});
    const d = await r.json();
    if (!r.ok) { log('✗ ' + d.detail, false); return; }
    log(`✓ Lưu: ${d.saved}  |  det_score=${d.det_score}  |  face=${d.face_w}×${d.face_h}px`);
    const img = document.getElementById('preview');
    img.src = '/snapshot?' + Date.now();
    img.style.display = 'block';
    loadList();
  } catch(e) { log('✗ ' + e, false); }
}
async function loadList() {
  const r = await fetch('/register/list');
  const d = await r.json();
  const area = document.getElementById('list-area');
  if (!d.persons.length) { area.innerHTML = '<i>Chưa có ai.</i>'; return; }
  let html = '<table><tr><th>Tên</th><th>Số ảnh</th><th>Files</th><th></th></tr>';
  for (const p of d.persons) {
    html += `<tr><td>${p.name}</td><td>${p.count}</td>
      <td style="color:#888;font-size:11px">${p.files.join(', ')}</td>
      <td><button class="btn-delete" onclick="deletePerson('${p.name}')">Xóa</button></td></tr>`;
  }
  html += '</table>';
  area.innerHTML = html;
}
async function deletePerson(name) {
  if (!confirm('Xóa toàn bộ ảnh của ' + name + '?')) return;
  const r = await fetch('/register/' + encodeURIComponent(name), {method:'DELETE'});
  const d = await r.json();
  log(`Đã xóa ${d.deleted} file của "${name}"`);
  loadList();
}
loadList();
</script>
</body>
</html>"""


@app.post("/capture/{name}")
def capture_face(name: str):
    """Chụp 1 frame, detect face, lưu embedding, reload DB ngay."""
    from fastapi import HTTPException
    with frame_lock:
        frame = latest_frame.copy() if latest_frame is not None else None
    if frame is None:
        raise HTTPException(503, detail="Camera chưa sẵn sàng")

    if roi_coords is not None:
        h, w = frame.shape[:2]
        rx1 = max(0, roi_coords[0]); ry1 = max(0, roi_coords[1])
        rx2 = min(w, roi_coords[2]); ry2 = min(h, roi_coords[3])
        roi = frame[ry1:ry2, rx1:rx2]
    else:
        roi = frame
    roi_resized = cv2.resize(roi, (640, 480), interpolation=cv2.INTER_LINEAR)

    faces = face_app.get(roi_resized)
    if not faces:
        raise HTTPException(400, detail="Không detect được mặt — đứng gần camera hơn")

    face   = max(faces, key=lambda f: f.det_score)
    fx1, fy1, fx2, fy2 = map(int, face.bbox)
    face_w = fx2 - fx1

    if face_w < MIN_FACE_PX:
        raise HTTPException(400, detail=f"Mặt quá nhỏ ({face_w}px) — đứng gần hơn hoặc chỉnh ROI")
    if face.det_score < 0.7:
        raise HTTPException(400, detail=f"det_score thấp ({face.det_score:.2f}) — cải thiện ánh sáng")

    os.makedirs(EMB_DIR, exist_ok=True)
    existing  = [f for f in os.listdir(EMB_DIR)
                 if f.startswith(name + "_") and f.endswith(".npy")]
    idx       = len(existing) + 1
    save_path = os.path.join(EMB_DIR, f"{name}_{idx}.npy")
    np.save(save_path, face.normed_embedding)
    reload_db()

    return {
        "saved"     : save_path,
        "det_score" : round(float(face.det_score), 3),
        "face_w"    : face_w,
        "face_h"    : fy2 - fy1,
        "total_imgs": idx,
    }


@app.get("/register/list")
def register_list():
    """Danh sách người đã đăng ký và số ảnh mỗi người."""
    persons = defaultdict(list)
    for fname in os.listdir(EMB_DIR):
        if not fname.endswith(".npy"):
            continue
        name = fname.rsplit("_", 1)[0]
        persons[name].append(fname)
    return {
        "persons": [
            {"name": name, "count": len(files), "files": sorted(files)}
            for name, files in sorted(persons.items())
        ]
    }


@app.delete("/register/{name}")
def delete_person(name: str):
    """Xóa toàn bộ embedding của 1 người và reload DB."""
    files = [f for f in os.listdir(EMB_DIR)
             if f.startswith(name + "_") and f.endswith(".npy")]
    for f in files:
        os.remove(os.path.join(EMB_DIR, f))
    reload_db()
    return {"deleted": len(files), "name": name}


# ─────────────────────────────────────────────
#  GHI VIDEO TEST DATASET
# ─────────────────────────────────────────────

recording_state = {
    "active"  : False,
    "writer"  : None,
    "filename": None,
}


@app.post("/record/start")
def record_start(video_name: str):
    """Bắt đầu ghi video từ camera vào file dataset_test/{video_name}.mp4"""
    from fastapi import HTTPException
    if recording_state["active"]:
        raise HTTPException(400, detail="Đang ghi rồi — gọi /record/stop trước")

    with frame_lock:
        frame = latest_frame.copy() if latest_frame is not None else None
    if frame is None:
        raise HTTPException(503, detail="Camera chưa sẵn sàng")

    filename = os.path.join(DATASET_DIR, video_name + ".mp4")
    os.makedirs(os.path.dirname(filename), exist_ok=True)

    h, w = frame.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    recording_state["writer"]   = cv2.VideoWriter(filename, fourcc, 20, (w, h))
    recording_state["filename"] = filename
    recording_state["active"]   = True

    def write_loop():
        while recording_state["active"]:
            with frame_lock:
                f = latest_frame.copy() if latest_frame is not None else None
            if f is not None:
                recording_state["writer"].write(f)
            time.sleep(1 / 20)

    threading.Thread(target=write_loop, daemon=True).start()
    return {"status": "recording", "file": filename}


@app.post("/record/stop")
def record_stop():
    """Dừng ghi và lưu file."""
    from fastapi import HTTPException
    if not recording_state["active"]:
        raise HTTPException(400, detail="Chưa bắt đầu ghi")

    recording_state["active"] = False
    time.sleep(0.2)
    recording_state["writer"].release()
    recording_state["writer"] = None
    return {"status": "saved", "file": recording_state["filename"]}


@app.get("/record/status")
def record_status():
    return {
        "active"  : recording_state["active"],
        "filename": recording_state["filename"],
    }

In [10]:
app = FastAPI()

# ─────────────────────────────────────────────
#  SNAPSHOT + ROI
# ─────────────────────────────────────────────

@app.get("/snapshot")
def snapshot():
    """Trả về 1 frame JPEG để UI dùng làm nền chọn ROI."""
    with frame_lock:
        frame = latest_frame.copy() if latest_frame is not None else None
    if frame is None:
        return JSONResponse({"error": "no frame"}, status_code=503)
    _, buf = cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 80])
    return StreamingResponse(iter([buf.tobytes()]), media_type="image/jpeg")


@app.get("/roi/select", response_class=HTMLResponse)
def roi_select_ui():
    """Giao diện web kéo-thả để chọn ROI trực tiếp trên frame."""
    return """<!DOCTYPE html>
<html>
<head>
<title>Chọn ROI</title>
<style>
    body { margin:0; background:#111; display:flex; flex-direction:column;
        align-items:center; padding:20px; font-family:monospace; color:#eee; }
    canvas { cursor:crosshair; border:2px solid #0f0; }
    #info  { margin:10px; font-size:14px; }
    button { margin:8px; padding:8px 20px; background:#0a0; color:#fff;
            border:none; cursor:pointer; font-size:14px; }
</style>
</head>
<body>
<h3>Kéo để chọn vùng ROI (cửa ra vào)</h3>
<canvas id="c"></canvas>
<div id="info">Đang tải frame...</div>
<button onclick="saveROI()">Lưu ROI</button>
<button onclick="location.reload()">Tải lại frame</button>
<script>
const canvas = document.getElementById('c');
const ctx    = canvas.getContext('2d');
let img = new Image();
let sx, sy, ex, ey, drawing = false;
let scaleX = 1, scaleY = 1;

img.onload = () => {
const maxW = Math.min(window.innerWidth - 40, 1200);
scaleX = img.naturalWidth  / Math.min(img.naturalWidth, maxW);
scaleY = img.naturalHeight / (img.naturalHeight * maxW / img.naturalWidth);
canvas.width  = Math.min(img.naturalWidth,  maxW);
canvas.height = img.naturalHeight * canvas.width / img.naturalWidth;
ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
document.getElementById('info').textContent =
    `Frame: ${img.naturalWidth}×${img.naturalHeight}`;
};
img.src = '/snapshot?' + Date.now();

canvas.addEventListener('mousedown', e => {
const r = canvas.getBoundingClientRect();
sx = e.clientX - r.left; sy = e.clientY - r.top; drawing = true;
});
canvas.addEventListener('mousemove', e => {
if (!drawing) return;
const r = canvas.getBoundingClientRect();
ex = e.clientX - r.left; ey = e.clientY - r.top;
ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
ctx.strokeStyle = '#0f0'; ctx.lineWidth = 2;
ctx.strokeRect(sx, sy, ex-sx, ey-sy);
});
canvas.addEventListener('mouseup', () => { drawing = false; });

function saveROI() {
if (sx === undefined) { alert('Chưa chọn vùng!'); return; }
const x1 = Math.round(Math.min(sx,ex)*scaleX), y1 = Math.round(Math.min(sy,ey)*scaleY);
const x2 = Math.round(Math.max(sx,ex)*scaleX), y2 = Math.round(Math.max(sy,ey)*scaleY);
fetch('/roi/save', {
    method:'POST', headers:{'Content-Type':'application/json'},
    body: JSON.stringify({x1,y1,x2,y2})
}).then(r=>r.json()).then(d => {
    document.getElementById('info').textContent =
    `Đã lưu ROI: [${d.x1}, ${d.y1}, ${d.x2}, ${d.y2}]`;
});
}
</script>
</body>
</html>"""


@app.post("/roi/save")
async def roi_save(request: Request):
    global roi_coords
    data = await request.json()
    roi_coords = [data["x1"], data["y1"], data["x2"], data["y2"]]
    np.save(ROI_PATH, np.array(roi_coords))
    return roi_coords


@app.get("/roi/current")
def roi_current():
    return {"roi": roi_coords}


# ─────────────────────────────────────────────
#  VIDEO STREAM
# ─────────────────────────────────────────────

def gen_frames():
    """Generator MJPEG — crop ROI, resize, detect, recognize, scale bbox về frame gốc."""
    while True:
        with frame_lock:
            frame = latest_frame.copy() if latest_frame is not None else None
        if frame is None:
            continue

        draw_frame = frame.copy()

        if roi_coords is not None:
            rx1, ry1, rx2, ry2 = roi_coords
            h, w = frame.shape[:2]
            rx1, ry1 = max(0, rx1), max(0, ry1)
            rx2, ry2 = min(w, rx2), min(h, ry2)

            roi          = frame[ry1:ry2, rx1:rx2]
            roi_resized  = cv2.resize(roi, (640, 480), interpolation=cv2.INTER_LINEAR)
            faces        = face_app.get(roi_resized)

            # Vẽ khung ROI
            cv2.rectangle(draw_frame, (rx1,ry1), (rx2,ry2), (255,165,0), 2)
            cv2.putText(draw_frame, "ROI", (rx1, ry1-6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,165,0), 1)

            for face in faces:
                if face.det_score < DET_THRESHOLD:
                    continue
                fx1, fy1, fx2, fy2 = map(int, face.bbox)
                if (fx2 - fx1) < MIN_FACE_PX:
                    continue
                name, score = recognize(face.normed_embedding)
                bx1,by1,bx2,by2 = scale_bbox_to_frame(fx1,fy1,fx2,fy2, 640, 480)
                color = (0,255,0) if name != "Unknown" else (0,0,255)
                cv2.rectangle(draw_frame, (bx1,by1), (bx2,by2), color, 2)
                cv2.putText(draw_frame, f"{name} {score:.2f}",
                            (bx1, by1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        else:
            faces = face_app.get(frame)
            for face in faces:
                if face.det_score < DET_THRESHOLD:
                    continue
                x1,y1,x2,y2 = map(int, face.bbox)
                name, score = recognize(face.normed_embedding)
                color = (0,255,0) if name != "Unknown" else (0,0,255)
                cv2.rectangle(draw_frame, (x1,y1), (x2,y2), color, 2)
                cv2.putText(draw_frame, f"{name} {score:.2f}",
                            (x1,y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            cv2.putText(draw_frame, "ROI chua duoc chon - vao /roi/select",
                        (20,40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,165,255), 2)

        # Overlay REC nếu đang ghi
        if recording_state["active"]:
            cv2.putText(draw_frame, "● REC",
                        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255), 2)
            cv2.putText(draw_frame, str(recording_state["filename"]),
                        (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 1)

        _, buf = cv2.imencode(".jpg", draw_frame, [cv2.IMWRITE_JPEG_QUALITY, 75])
        yield (b"--frame\r\nContent-Type: image/jpeg\r\n\r\n"
            + buf.tobytes() + b"\r\n")


@app.get("/video")
def video():
    return StreamingResponse(gen_frames(),
                            media_type="multipart/x-mixed-replace; boundary=frame")


# ─────────────────────────────────────────────
#  ĐĂNG KÝ KHUÔN MẶT
# ─────────────────────────────────────────────

@app.get("/register", response_class=HTMLResponse)
def register_ui():
    """Web UI đăng ký khuôn mặt nhân viên."""
    return """<!DOCTYPE html>
<html>
<head>
<title>Đăng ký khuôn mặt</title>
<style>
    * { box-sizing:border-box; margin:0; padding:0; }
    body { background:#0d0d0d; color:#eee; font-family:monospace;
        display:flex; flex-direction:column; align-items:center; padding:24px; }
    h2   { margin-bottom:16px; color:#0f0; }
    .card { background:#1a1a1a; border:1px solid #333; border-radius:8px;
            padding:20px; width:100%; max-width:720px; margin-bottom:16px; }
    input { width:100%; padding:8px; background:#222; border:1px solid #555;
            color:#eee; border-radius:4px; font-size:15px; }
    .row  { display:flex; gap:10px; margin-top:10px; }
    button { flex:1; padding:10px; border:none; border-radius:4px;
            cursor:pointer; font-size:14px; font-family:monospace; }
    .btn-capture { background:#0a0; color:#fff; }
    .btn-reload  { background:#555; color:#fff; }
    .btn-delete  { background:#a00; color:#fff; }
    #preview { width:100%; border-radius:4px; margin-top:12px;
            border:1px solid #333; display:none; }
    #log { background:#111; border:1px solid #333; border-radius:4px;
        padding:12px; font-size:13px; max-height:220px;
        overflow-y:auto; white-space:pre-wrap; margin-top:12px; }
    .ok  { color:#0f0; } .err { color:#f55; }
    table { width:100%; border-collapse:collapse; font-size:13px; }
    th,td { padding:6px 10px; border-bottom:1px solid #2a2a2a; text-align:left; }
    th { color:#0f0; }
</style>
</head>
<body>
<h2>Đăng ký khuôn mặt nhân viên</h2>
<div class="card">
    <b>Bước 1 — Đứng trước camera, điền tên rồi bấm Chụp (3–5 lần, đổi góc mỗi lần)</b>
    <div class="row" style="margin-top:10px">
    <input id="name" placeholder="Nhập tên nhân viên (vd: anhht)" />
    </div>
    <div class="row">
    <button class="btn-capture" onclick="capture()">📸 Chụp ảnh</button>
    <button class="btn-reload"  onclick="loadList()">🔄 Tải danh sách</button>
    </div>
    <img id="preview" />
    <div id="log">Sẵn sàng...</div>
</div>
<div class="card">
    <b>Danh sách đã đăng ký</b>
    <div id="list-area" style="margin-top:10px">Bấm 🔄 để tải...</div>
</div>
<script>
function log(msg, ok=true) {
const el = document.getElementById('log');
const line = document.createElement('div');
line.className = ok ? 'ok' : 'err';
line.textContent = new Date().toLocaleTimeString() + '  ' + msg;
el.appendChild(line);
el.scrollTop = el.scrollHeight;
}
async function capture() {
const name = document.getElementById('name').value.trim();
if (!name) { log('⚠ Chưa nhập tên!', false); return; }
log(`Đang chụp cho "${name}"...`);
try {
    const r = await fetch('/capture/' + encodeURIComponent(name), {method:'POST'});
    const d = await r.json();
    if (!r.ok) { log('✗ ' + d.detail, false); return; }
    log(`✓ Lưu: ${d.saved}  |  det_score=${d.det_score}  |  face=${d.face_w}×${d.face_h}px`);
    const img = document.getElementById('preview');
    img.src = '/snapshot?' + Date.now();
    img.style.display = 'block';
    loadList();
} catch(e) { log('✗ ' + e, false); }
}
async function loadList() {
const r = await fetch('/register/list');
const d = await r.json();
const area = document.getElementById('list-area');
if (!d.persons.length) { area.innerHTML = '<i>Chưa có ai.</i>'; return; }
let html = '<table><tr><th>Tên</th><th>Số ảnh</th><th>Files</th><th></th></tr>';
for (const p of d.persons) {
    html += `<tr><td>${p.name}</td><td>${p.count}</td>
    <td style="color:#888;font-size:11px">${p.files.join(', ')}</td>
    <td><button class="btn-delete" onclick="deletePerson('${p.name}')">Xóa</button></td></tr>`;
}
html += '</table>';
area.innerHTML = html;
}
async function deletePerson(name) {
if (!confirm('Xóa toàn bộ ảnh của ' + name + '?')) return;
const r = await fetch('/register/' + encodeURIComponent(name), {method:'DELETE'});
const d = await r.json();
log(`Đã xóa ${d.deleted} file của "${name}"`);
loadList();
}
loadList();
</script>
</body>
</html>"""


@app.post("/capture/{name}")
def capture_face(name: str):
    """Chụp 1 frame, detect face, lưu embedding, reload DB ngay."""
    from fastapi import HTTPException
    with frame_lock:
        frame = latest_frame.copy() if latest_frame is not None else None
    if frame is None:
        raise HTTPException(503, detail="Camera chưa sẵn sàng")

    if roi_coords is not None:
        h, w = frame.shape[:2]
        rx1 = max(0, roi_coords[0]); ry1 = max(0, roi_coords[1])
        rx2 = min(w, roi_coords[2]); ry2 = min(h, roi_coords[3])
        roi = frame[ry1:ry2, rx1:rx2]
    else:
        roi = frame
    roi_resized = cv2.resize(roi, (640, 480), interpolation=cv2.INTER_LINEAR)

    faces = face_app.get(roi_resized)
    if not faces:
        raise HTTPException(400, detail="Không detect được mặt — đứng gần camera hơn")

    face   = max(faces, key=lambda f: f.det_score)
    fx1, fy1, fx2, fy2 = map(int, face.bbox)
    face_w = fx2 - fx1

    if face_w < MIN_FACE_PX:
        raise HTTPException(400, detail=f"Mặt quá nhỏ ({face_w}px) — đứng gần hơn hoặc chỉnh ROI")
    if face.det_score < 0.7:
        raise HTTPException(400, detail=f"det_score thấp ({face.det_score:.2f}) — cải thiện ánh sáng")

    os.makedirs(EMB_DIR, exist_ok=True)
    existing  = [f for f in os.listdir(EMB_DIR)
                if f.startswith(name + "_") and f.endswith(".npy")]
    idx       = len(existing) + 1
    save_path = os.path.join(EMB_DIR, f"{name}_{idx}.npy")
    np.save(save_path, face.normed_embedding)
    reload_db()

    return {
        "saved"     : save_path,
        "det_score" : round(float(face.det_score), 3),
        "face_w"    : face_w,
        "face_h"    : fy2 - fy1,
        "total_imgs": idx,
    }


@app.get("/register/list")
def register_list():
    """Danh sách người đã đăng ký và số ảnh mỗi người."""
    persons = defaultdict(list)
    for fname in os.listdir(EMB_DIR):
        if not fname.endswith(".npy"):
            continue
        name = fname.rsplit("_", 1)[0]
        persons[name].append(fname)
    return {
        "persons": [
            {"name": name, "count": len(files), "files": sorted(files)}
            for name, files in sorted(persons.items())
        ]
    }


@app.delete("/register/{name}")
def delete_person(name: str):
    """Xóa toàn bộ embedding của 1 người và reload DB."""
    files = [f for f in os.listdir(EMB_DIR)
            if f.startswith(name + "_") and f.endswith(".npy")]
    for f in files:
        os.remove(os.path.join(EMB_DIR, f))
    reload_db()
    return {"deleted": len(files), "name": name}


# ─────────────────────────────────────────────
#  GHI VIDEO TEST DATASET
# ─────────────────────────────────────────────

recording_state = {
    "active"  : False,
    "writer"  : None,
    "filename": None,
}


@app.post("/record/start")
def record_start(video_name: str):
    """Bắt đầu ghi video từ camera vào file dataset_test/{video_name}.mp4"""
    from fastapi import HTTPException
    if recording_state["active"]:
        raise HTTPException(400, detail="Đang ghi rồi — gọi /record/stop trước")

    with frame_lock:
        frame = latest_frame.copy() if latest_frame is not None else None
    if frame is None:
        raise HTTPException(503, detail="Camera chưa sẵn sàng")

    filename = os.path.join(DATASET_DIR, video_name + ".mp4")
    os.makedirs(os.path.dirname(filename), exist_ok=True)

    h, w = frame.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    recording_state["writer"]   = cv2.VideoWriter(filename, fourcc, 20, (w, h))
    recording_state["filename"] = filename
    recording_state["active"]   = True

    def write_loop():
        while recording_state["active"]:
            with frame_lock:
                f = latest_frame.copy() if latest_frame is not None else None
            if f is not None:
                recording_state["writer"].write(f)
            time.sleep(1 / 20)

    threading.Thread(target=write_loop, daemon=True).start()
    return {"status": "recording", "file": filename}


@app.post("/record/stop")
def record_stop():
    """Dừng ghi và lưu file."""
    from fastapi import HTTPException
    if not recording_state["active"]:
        raise HTTPException(400, detail="Chưa bắt đầu ghi")

    recording_state["active"] = False
    time.sleep(0.2)
    recording_state["writer"].release()
    recording_state["writer"] = None
    return {"status": "saved", "file": recording_state["filename"]}


@app.get("/record/status")
def record_status():
    return {
        "active"  : recording_state["active"],
        "filename": recording_state["filename"],
    }

In [ ]:
import uvicorn
config = uvicorn.Config(app, host="0.0.0.0", port=8501)
server = uvicorn.Server(config)
await server.serve()
# ─────────────────────────────────────────
# Endpoints:
#   /roi/select      → Chọn ROI bằng kéo chuột
#   /video           → MJPEG stream nhận diện realtime
#   /register        → Web UI đăng ký khuôn mặt
#   /register/list   → Danh sách đã đăng ký (JSON)
#   /capture/{name}  → Chụp 1 ảnh đăng ký (POST)
#   /register/{name} → Xóa người (DELETE)
#   /record/start    → Bắt đầu ghi video test (?video_name=...)
#   /record/stop     → Dừng ghi
#   /record/status   → Trạng thái ghi
#   /snapshot        → Lấy 1 frame JPEG
#   /roi/current     → Tọa độ ROI hiện tại
# ─────────────────────────────────────────

INFO:     Started server process [392931]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8501 (Press CTRL+C to quit)


INFO:     192.168.1.1:53003 - "GET /video HTTP/1.1" 200 OK


/home/anhht/miniconda3/envs/face_attendance/lib/python3.9/site-packages/insightface/utils/transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4


In [10]:
import os, csv

DATASET_DIR = "dataset_test"
LABELS_PATH = "dataset_test/labels.csv"

rows = []



for root, dirs, files in os.walk(DATASET_DIR):
    dirs[:] = [d for d in dirs if not d.startswith(".")]
    for fname in sorted(files):
        if not fname.endswith(".mp4"):
            continue

        full_path = os.path.join(root, fname)
        rel_path  = os.path.relpath(full_path, DATASET_DIR)  # employees/anhht/normal.mp4

        parts = rel_path.replace("\\", "/").split("/")
        # parts = ["employees", "anhht", "normal.mp4"]  hoặc ["unknown", "stranger_1.mp4"]

        if parts[0] == "unknown":
            label = "Unknown"
        elif parts[0] == "employees" and len(parts) >= 3:
            label = parts[1]   # tên người = thư mục con của employees/
        else:
            print(f"[SKIP] Không parse được: {rel_path}")
            continue

        rows.append([rel_path.replace("\\", "/"), label])
        print(f"  {rel_path:50s} → {label}")

os.makedirs(DATASET_DIR, exist_ok=True)
with open(LABELS_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["video", "label"])
    writer.writerows(rows)

print(f"\n✓ Tạo {LABELS_PATH} — {len(rows)} video")

  employees/anhht/normal.mp4                         → anhht

✓ Tạo dataset_test/labels.csv — 1 video


In [11]:
import os, csv
import cv2
import numpy as np
import pandas as pd
from collections import defaultdict

DATASET_DIR   = "dataset_test"
LABELS_PATH   = "dataset_test/labels.csv"
PRED_PATH     = "dataset_test/predictions.csv"
SAMPLE_STEP   = 15   # lấy 1 frame mỗi 0.6 giây (25FPS) — đủ độc lập

# ── Đọc labels ──
labels_df = pd.read_csv(LABELS_PATH)
print(f"Tổng số video: {len(labels_df)}")
print(labels_df["label"].value_counts().to_string())

# ── Chạy pipeline trên từng video ──
frame_rows   = []   # kết quả frame-level
event_rows   = []   # kết quả event-level (1 video = 1 dự đoán)

for _, row in labels_df.iterrows():
    video_path = os.path.join(DATASET_DIR, row["video"])
    gt_label   = row["label"]

    if not os.path.exists(video_path):
        print(f"[SKIP] Không tìm thấy: {video_path}")
        continue

    cap       = cv2.VideoCapture(video_path)
    frame_idx = 0
    preds_this_video = []   # predictions trong video này

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Chỉ lấy mỗi SAMPLE_STEP frame
        if frame_idx % SAMPLE_STEP != 0:
            frame_idx += 1
            continue

        # Crop ROI nếu có
        if roi_coords is not None:
            h, w = frame.shape[:2]
            rx1 = max(0, roi_coords[0]); ry1 = max(0, roi_coords[1])
            rx2 = min(w, roi_coords[2]); ry2 = min(h, roi_coords[3])
            proc = cv2.resize(frame[ry1:ry2, rx1:rx2], (640, 480),
                              interpolation=cv2.INTER_LINEAR)
        else:
            proc = frame

        faces = face_app.get(proc)

        if faces:
            face = max(faces, key=lambda f: f.det_score)
            if face.det_score >= DET_THRESHOLD:
                fx1, fy1, fx2, fy2 = map(int, face.bbox)
                if (fx2 - fx1) >= MIN_FACE_PX:
                    name, score = recognize(face.normed_embedding)
                    preds_this_video.append(name)
                    frame_rows.append({
                        "video"       : row["video"],
                        "frame"       : frame_idx,
                        "prediction"  : name,
                        "score"       : round(score, 4),
                        "ground_truth": gt_label,
                    })

        frame_idx += 1

    cap.release()

    # Event-level: majority vote trên các frame detect được
    if preds_this_video:
        event_pred = max(set(preds_this_video), key=preds_this_video.count)
        confidence = preds_this_video.count(event_pred) / len(preds_this_video)
    else:
        event_pred  = "Unknown"
        confidence  = 0.0

    match = "✓" if event_pred == gt_label else "✗"
    print(f"{match} {row['video']:50s} GT={gt_label:15s} Pred={event_pred:15s} "
          f"({len(preds_this_video)} frames, conf={confidence:.0%})")

    event_rows.append({
        "video"       : row["video"],
        "ground_truth": gt_label,
        "prediction"  : event_pred,
        "n_frames"    : len(preds_this_video),
        "confidence"  : round(confidence, 3),
    })

# ── Lưu predictions frame-level ──
pd.DataFrame(frame_rows).to_csv(PRED_PATH, index=False)
print(f"\n✓ Lưu {PRED_PATH} — {len(frame_rows)} frame")

# ── Tính metric ──
event_df = pd.DataFrame(event_rows)
y_true   = event_df["ground_truth"]
y_pred   = event_df["prediction"]

all_labels = sorted(set(y_true) | set(y_pred))

# Confusion matrix thủ công
print("\n" + "="*60)
print("CONFUSION MATRIX (event-level, 1 video = 1 dự đoán)")
print("="*60)
# header = f"{'GT \\ Pred':15s}" + "".join(f"{l:15s}" for l in all_labels)
header = "GT \\ Pred".ljust(15) + "".join(f"{l:15s}" for l in all_labels)
print(header)
for gt in all_labels:
    row_str = f"{gt:15s}"
    for pred in all_labels:
        count = ((y_true == gt) & (y_pred == pred)).sum()
        row_str += f"{count:<15d}"
    print(row_str)

# Precision / Recall / F1 per class
print("\n" + "="*60)
print("METRIC PER CLASS (event-level)")
print("="*60)
print(f"{'Class':15s} {'Precision':>10s} {'Recall':>10s} {'F1':>10s} {'Support':>10s}")
for cls in all_labels:
    tp = ((y_true == cls) & (y_pred == cls)).sum()
    fp = ((y_true != cls) & (y_pred == cls)).sum()
    fn = ((y_true == cls) & (y_pred != cls)).sum()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0.0
    sup  = (y_true == cls).sum()
    print(f"{cls:15s} {prec:>10.4f} {rec:>10.4f} {f1:>10.4f} {sup:>10d}")

# FAR / FRR
unknown_mask = (y_true == "Unknown")
FAR = ((y_pred != "Unknown") &  unknown_mask).sum() / unknown_mask.sum() if unknown_mask.sum() > 0 else 0.0
FRR = ((y_pred == "Unknown") & ~unknown_mask).sum() / (~unknown_mask).sum() if (~unknown_mask).sum() > 0 else 0.0

print("\n" + "="*60)
print("TỔNG KẾT")
print("="*60)
total   = len(event_df)
correct = (y_true == y_pred).sum()
print(f"Accuracy (event-level) : {correct}/{total} = {correct/total:.4f}")
print(f"FAR (nhận nhầm người lạ): {FAR:.4f}  ({FAR:.2%})")
print(f"FRR (từ chối người thật): {FRR:.4f}  ({FRR:.2%})")
print(f"\nSIM_THRESHOLD hiện tại : {SIM_THRESHOLD}")
print(f"DET_THRESHOLD hiện tại  : {DET_THRESHOLD}")

Tổng số video: 1
label
anhht    1


/home/anhht/miniconda3/envs/face_attendance/lib/python3.9/site-packages/insightface/utils/transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4


✓ employees/anhht/normal.mp4                         GT=anhht           Pred=anhht           (11 frames, conf=91%)

✓ Lưu dataset_test/predictions.csv — 11 frame

CONFUSION MATRIX (event-level, 1 video = 1 dự đoán)
GT \ Pred      anhht          
anhht          1              

METRIC PER CLASS (event-level)
Class            Precision     Recall         F1    Support
anhht               1.0000     1.0000     1.0000          1

TỔNG KẾT
Accuracy (event-level) : 1/1 = 1.0000
FAR (nhận nhầm người lạ): 0.0000  (0.00%)
FRR (từ chối người thật): 0.0000  (0.00%)

SIM_THRESHOLD hiện tại : 0.3
DET_THRESHOLD hiện tại  : 0.6
